##### The goal of this notebook is to scrape **Tafseer Al-Jalalayn** from the website

`https://www.altafsir.com`

The link of the website for each Ayah follows this format

`https://www.altafsir.com/Tafasir.asp?tMadhNo=0&tTafsirNo=74&tSoraNo={Surrah_Number}&tAyahNo={Ayah_Number}&tDisplay=yes&UserProfile=0&LanguageId=2`


In [3]:
pip install selenium

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.7 MB 3.6 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.7 MB 3.7 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.7 MB 3.8 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.7 MB 3.8 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 3.9 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.7 MB 3.9 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.7 MB 3.5 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.7 MB 3.5 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.7 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.7 MB 2.6 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.7 MB 2.6 MB/s eta 0:00:02
   ---------------------- 


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from tqdm import tqdm
import time
import os
import json
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
quran = pd.read_csv("Datasets\quran_surahs.csv")
quran

,Surah Number,Surah Name,Number of Ayat
0,1,Al-Fatiha,7
1,2,Al-Baqarah,286
2,3,Al-Imran,200
3,4,An-Nisa,176
4,5,Al-Ma'idah,120
...,...,...,...
109,110,An-Nasr,3
110,111,Al-Masad,5
111,112,Al-Ikhlas,4
112,113,Al-Falaq,5


In [28]:
# Trying scraping on a single ayah

options = Options()
options.headless = True
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)

url = 'https://www.altafsir.com/Tafasir.asp?tMadhNo=0&tTafsirNo=74&tSoraNo=1&tAyahNo=1&tDisplay=yes&UserProfile=0&LanguageId=2'

driver = webdriver.Chrome(options=options)
driver.get(url)

tafseer = driver.find_element(By.XPATH, '//*[@id="SearchResults"]/div/font/font')
tafseer.text

'In the Name of God the Compassionate the Merciful'

In [38]:
# Trying scraping on a single ayah

options = Options()
options.headless = True
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
a = 1
b = 3
url = f'https://www.altafsir.com/Tafasir.asp?tMadhNo=0&tTafsirNo=74&tSoraNo={a}&tAyahNo={b}&tDisplay=yes&UserProfile=0&LanguageId=2'
driver.get(url)

tafseer = driver.find_element(By.XPATH, '//*[@id="SearchResults"]/div/font/font')
print(tafseer.text)

driver.quit()

The Compassionate the Merciful that is to say the One who possesses ‘mercy’ which means to want what is good for those who deserve it.


In [45]:
results = []
options = Options()
options.headless = True
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
for _, surah in tqdm(quran.iterrows(), total=len(quran), desc="Surahs"):
    surah_number = surah["Surah Number"]
    num_ayat = surah["Number of Ayat"]

    for ayah_number in range(1, num_ayat + 1):
        url = f'https://www.altafsir.com/Tafasir.asp?tMadhNo=0&tTafsirNo=74&tSoraNo={surah_number}&tAyahNo={ayah_number}&tDisplay=yes&UserProfile=0&LanguageId=2'
        driver.get(url)

        tafseer = driver.find_element(By.XPATH, '//*[@id="SearchResults"]/div/font/font')
        results.append({
            "Surah Number": surah_number,
            "Ayah Number": ayah_number,
            "Text": tafseer.text
        })
driver.quit()

Surahs: 100%|██████████| 114/114 [2:54:59<00:00, 92.10s/it]  


In [46]:
df_results = pd.DataFrame(results)
df_results.to_csv("tafseer_scraped.csv", index=False)

In [55]:
df_results

,Surah Number,Ayah Number,Text
0,1,1,In the Name of God the Compassionate the Merciful
1,1,2,In the Name of God the name of a thing is that...
2,1,3,The Compassionate the Merciful that is to say ...
3,1,4,Master of the Day of Judgement that is the day...
4,1,5,You alone we worship and You alone we ask for ...
...,...,...,...
6231,114,2,the King of mankind
6232,114,3,the God of mankind both maliki’l-nās and ilāhi...
6233,114,4,from the evil of the slinking whisperer Satan ...
6234,114,5,who whispers in the breasts of mankind in thei...


### **Now I will try to scrape Tafseer al Jalaleen from this website**
`https://quran.com/`

`https://quran.com/{Surah_number}:{Ayah_number}/tafsirs/en-tafisr-ibn-kathir`


In [28]:
options = Options()
options.headless = True
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)

url = 'https://quran.com/1:7/tafsirs/en-tafisr-ibn-kathir'

In [32]:
wait = WebDriverWait(driver, 25)

tafseer_el = wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'div[class*="TafsirView"] div[class*="TafsirText_md"]')
    )
)

tafseer_text = tafseer_el.text.strip()
tafseer_text


'We mentioned the Hadith in which the servant proclaims,\nاهْدِنَا الصِّرَاطَ الْمُسْتَقِيمَ\n(Guide us to the straight way) and Allah says, "This is for My servant, and My servant shall acquire what he asks for." Allah\'s statement.\nصِرَاطَ الَّذِينَ أَنْعَمْتَ عَلَيْهِمْ\n(The way of those upon whom You have bestowed Your grace) defines the path. `Those upon whom Allah has bestowed His grace\' are those mentioned in Surat An-Nisa\' (chapter 4), when Allah said,\nوَمَن يُطِعِ اللَّهَ وَالرَّسُولَ فَأُوْلَـئِكَ مَعَ الَّذِينَ أَنْعَمَ اللَّهُ عَلَيْهِم مِّنَ النَّبِيِّينَ وَالصِّدِّيقِينَ وَالشُّهَدَآءِ وَالصَّـلِحِينَ وَحَسُنَ أُولَـئِكَ رَفِيقاً - ذلِكَ الْفَضْلُ مِنَ اللَّهِ وَكَفَى بِاللَّهِ عَلِيماً\n(And whoever obeys Allah and the Messenger (Muhammad ﷺ ), then they will be in the company of those on whom Allah has bestowed His grace, the Prophets, the Siddiqin (the truly faithful), the martyrs, and the righteous. And how excellent these companions are! Such is the bounty from A

In [27]:
driver = webdriver.Chrome(options=options)
driver.get(url)

tafseer = driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[3]/div/div[3]/div')
tafseer.text

'صراط الذين انعمت عليهم غير المغضوب عليهم ولا الضالين ٧\nﭫ\nﭬ\nﭭ\nﭮ\nﭯ\nﭰ\nﭱ\nﭲ\nﭳ\nﭴ\n3\nWe mentioned the Hadith in which the servant proclaims,\nاهْدِنَا الصِّرَاطَ الْمُسْتَقِيمَ\n(Guide us to the straight way) and Allah says, "This is for My servant, and My servant shall acquire what he asks for." Allah\'s statement.\nصِرَاطَ الَّذِينَ أَنْعَمْتَ عَلَيْهِمْ\n(The way of those upon whom You have bestowed Your grace) defines the path. `Those upon whom Allah has bestowed His grace\' are those mentioned in Surat An-Nisa\' (chapter 4), when Allah said,\nوَمَن يُطِعِ اللَّهَ وَالرَّسُولَ فَأُوْلَـئِكَ مَعَ الَّذِينَ أَنْعَمَ اللَّهُ عَلَيْهِم مِّنَ النَّبِيِّينَ وَالصِّدِّيقِينَ وَالشُّهَدَآءِ وَالصَّـلِحِينَ وَحَسُنَ أُولَـئِكَ رَفِيقاً - ذلِكَ الْفَضْلُ مِنَ اللَّهِ وَكَفَى بِاللَّهِ عَلِيماً\n(And whoever obeys Allah and the Messenger (Muhammad ﷺ ), then they will be in the company of those on whom Allah has bestowed His grace, the Prophets, the Siddiqin (the truly faithful), the mart

In [7]:
folder_name = "ayat"
os.makedirs(folder_name, exist_ok=True)

options = Options()
options.headless = True
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
for _, surah in tqdm(quran.iterrows() , total=len(quran), desc="Surahs"):
    surah_number = surah["Surah Number"]
    surah_number = surah["Surah Number"]
    num_ayat = surah["Number of Ayat"]

    for ayah_number in range(1, num_ayat + 1):
        url = f'https://quran.com/{surah_number}:{ayah_number}/tafsirs/en-tafisr-ibn-kathir'
        driver.get(url)

        wait = WebDriverWait(driver, 25)

        tafseer_el = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'div[class*="TafsirView"] div[class*="TafsirText_md"]')
            )
        )

        tafseer_text = tafseer_el.text.strip()

        res = {
            "Surah Number": surah_number,
            "Ayah Number": ayah_number,
            "Text": tafseer_text
        }

        time.sleep(1)
        filename = f"{surah_number}_{ayah_number}.txt"
        filepath = os.path.join(folder_name, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(res, f, ensure_ascii=False, indent=2)
driver.quit()

Surahs: 100%|██████████| 114/114 [16:06<00:00,  8.48s/it]


In [23]:
folder_path = "ayat"
all_data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
                all_data.append(data)
            except json.JSONDecodeError:
                print(f"Skipping {filename}, not valid JSON")

df = pd.DataFrame(all_data)
df = df.sort_values(["Surah Number", "Ayah Number"])

df.to_csv("combined_tafseer.csv", index=False, encoding="utf-8")
print("Saved combined_tafseer.csv successfully!")


Saved combined_tafseer.csv successfully!


In [24]:
df

,Surah Number,Ayah Number,Text
1074,1,1,Introduction to Fatihah\nWhich was revealed in...
1075,1,2,The Meaning of Al-Hamd\nAbu Ja`far bin Jarir s...
1076,1,3,"Allah said next,\nالرَّحْمَـنِ الرَّحِيمِ\n(Ar..."
1077,1,4,Indicating Sovereignty on the Day of Judgment\...
1078,1,5,The Linguistic and Religious Meaning of `Ibada...
...,...,...,...
194,114,2,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
195,114,3,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
196,114,4,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
197,114,5,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
